# P3: Acquisition & Integration — DSA 405 (Fall 2026)
**Unity ID:** nmbrockm  
**Research question:** Do S&P 500 companies whose 10-K Item 1A risk-factor sections get longer year over year have worse returns over the following 12 months?

This notebook follows my A8 plan more closely than my earlier company-metadata-only demo. It is a **pilot** using five companies and filings from 2019–2024. I will expand the sample after checking the extraction quality. Historical prices are only integrated when I provide a legally obtained CSV; I have **not** verified permission to automate Stooq downloads. Do not claim the price join is complete unless the final checks run.

## 1. Acquisition setup
I use the Wikipedia S&P 500 list for company identifiers and the SEC submissions API for 10-K filing dates and accession numbers. I then try to download the actual 10-K filing documents to measure Item 1A length. Raw responses are cached without changes. I wait at least one second between requests, identify my academic purpose in the User-Agent, and stop on HTTP errors. **Before running SEC requests, replace the contact email below with an email address you control.**

In [1]:
from pathlib import Path
from urllib.parse import urljoin
import io, re, time, json, html
import requests
import pandas as pd
from bs4 import BeautifulSoup

RAW = Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
PROCESSED = Path('data/processed'); PROCESSED.mkdir(parents=True, exist_ok=True)
CONTACT_EMAIL = 'nmbrockm@ncsu.edu'
N_COMPANIES = 5
START_YEAR, END_YEAR = 2019, 2024
if CONTACT_EMAIL.startswith('REPLACE_'):
    raise ValueError('Enter your real contact email in CONTACT_EMAIL before running.')
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': f'NCSU DSA405 student research ({CONTACT_EMAIL})', 'Accept-Encoding':'gzip, deflate'})
_last_request = [0.0]

def fetch_cached(url, cache_path, session=SESSION):
    cache_path = Path(cache_path)
    if cache_path.exists():
        print('CACHE:', cache_path)
        return cache_path.read_bytes()
    wait = max(0, 1.1 - (time.monotonic() - _last_request[0]))
    if wait: time.sleep(wait)
    try:
        response = session.get(url, timeout=45)
        _last_request[0] = time.monotonic()
        print('HTTP', response.status_code, url)
        print('Rate limit headers:', {k:v for k,v in response.headers.items() if 'rate' in k.lower()} or 'not supplied')
        response.raise_for_status()
        data = response.content
        if not data: raise ValueError('Empty response: '+url)
    except requests.RequestException as error:
        raise RuntimeError(f'Acquisition failed for {url}: {error}') from error
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_bytes(data)
    return data

### Deliberate acquisition failure test
This fake response lets me prove that an HTTP error stops the workflow, without making a bad request to the SEC.

In [2]:
class FakeResponse:
    status_code = 503
    headers = {}
    content = b'Unavailable'
    def raise_for_status(self):
        raise requests.HTTPError('503 Service Unavailable')
class FakeSession:
    def get(self, *args, **kwargs): return FakeResponse()
try:
    fetch_cached('https://example.invalid/test', RAW / '_never_created_test.bin', session=FakeSession())
    raise AssertionError('Failure was not caught')
except RuntimeError:
    print('PASS: simulated HTTP 503 stopped acquisition')

HTTP 503 https://example.invalid/test
Rate limit headers: not supplied
PASS: simulated HTTP 503 stopped acquisition


### S&P 500 company identifiers
I keep one row per CIK because some companies have multiple share classes. The current S&P 500 membership list is **not** a historical 2019 membership list, which creates survivorship bias.

In [3]:
wiki_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
wiki_html = fetch_cached(wiki_url, RAW / 'wikipedia_sp500.html')
tables = pd.read_html(io.BytesIO(wiki_html), match='CIK')
wiki = next((t for t in tables if 'CIK' in t.columns and 'Symbol' in t.columns), None)
if wiki is None: raise ValueError('Could not find the S&P 500 constituents table')
wiki['CIK'] = pd.to_numeric(wiki['CIK'], errors='raise').astype('int64')
companies = wiki.sort_values('Symbol').drop_duplicates('CIK').copy()
companies = companies[['CIK','Symbol','Security']].rename(columns={'Symbol':'ticker','Security':'company'})
pilot = companies.head(N_COMPANIES).reset_index(drop=True)
print('Security rows:', len(wiki), '| unique CIKs:', len(companies))
display(pilot)

HTTP 200 https://en.wikipedia.org/wiki/List_of_S%26P_500_companies
Rate limit headers: not supplied
Security rows: 503 | unique CIKs: 500


,CIK,ticker,company
0,1090872,A,Agilent Technologies
1,320193,AAPL,Apple Inc.
2,1551152,ABBV,AbbVie
3,1559720,ABNB,Airbnb
4,1800,ABT,Abbott Laboratories


### SEC 10-K filing metadata
I get annual filings using the SEC submissions API. Some older filings may be in additional submissions JSON files, so I follow those references too. Each accession number identifies a filing. Filing years here are based on the **filing date**, not necessarily the company's fiscal year.

In [4]:
def submission_frame(obj):
    recent = obj.get('filings', {}).get('recent', {})
    if not recent: return pd.DataFrame()
    return pd.DataFrame(recent)

filings = []
for rec in pilot.itertuples(index=False):
    cik = int(rec.CIK)
    root = f'https://data.sec.gov/submissions/CIK{cik:010d}.json'
    obj = json.loads(fetch_cached(root, RAW / f'sec_CIK{cik:010d}.json'))
    parts = [submission_frame(obj)]
    for extra in obj.get('filings', {}).get('files', []):
        # Older years may be in these additional submission files.
        name = extra['name']
        extra_obj = json.loads(fetch_cached('https://data.sec.gov/submissions/'+name, RAW / f'sec_{name}'))
        parts.append(pd.DataFrame(extra_obj))
    sub = pd.concat(parts, ignore_index=True)
    sub = sub[sub['form'].eq('10-K')].copy()
    sub['filingDate'] = pd.to_datetime(sub['filingDate'], errors='coerce')
    sub = sub[sub['filingDate'].dt.year.between(START_YEAR, END_YEAR)]
    for r in sub.itertuples(index=False):
        filings.append({'CIK':cik,'ticker':rec.ticker,'company':rec.company,
                        'filing_date':r.filingDate,'filing_year':r.filingDate.year,
                        'accession':r.accessionNumber,'primary_document':r.primaryDocument})
filings = pd.DataFrame(filings)
if filings.empty: raise ValueError('No 10-K filings found in the selected years')
filings = filings.sort_values(['CIK','filing_date']).drop_duplicates(['CIK','filing_year'], keep='last')
print('Selected company-year filings:',len(filings))
display(filings.head(10))

HTTP 200 https://data.sec.gov/submissions/CIK0001090872.json
Rate limit headers: not supplied
HTTP 200 https://data.sec.gov/submissions/CIK0001090872-submissions-001.json
Rate limit headers: not supplied


/tmp/ipykernel_545/1810757003.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sub = pd.concat(parts, ignore_index=True)


HTTP 200 https://data.sec.gov/submissions/CIK0000320193.json
Rate limit headers: not supplied
HTTP 200 https://data.sec.gov/submissions/CIK0000320193-submissions-001.json
Rate limit headers: not supplied


/tmp/ipykernel_545/1810757003.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sub = pd.concat(parts, ignore_index=True)


HTTP 200 https://data.sec.gov/submissions/CIK0001551152.json
Rate limit headers: not supplied
HTTP 200 https://data.sec.gov/submissions/CIK0001551152-submissions-001.json
Rate limit headers: not supplied


/tmp/ipykernel_545/1810757003.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sub = pd.concat(parts, ignore_index=True)


HTTP 200 https://data.sec.gov/submissions/CIK0001559720.json
Rate limit headers: not supplied
HTTP 200 https://data.sec.gov/submissions/CIK0001559720-submissions-001.json
Rate limit headers: not supplied


/tmp/ipykernel_545/1810757003.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sub = pd.concat(parts, ignore_index=True)


HTTP 200 https://data.sec.gov/submissions/CIK0000001800.json
Rate limit headers: not supplied
HTTP 200 https://data.sec.gov/submissions/CIK0000001800-submissions-001.json
Rate limit headers: not supplied
HTTP 200 https://data.sec.gov/submissions/CIK0000001800-submissions-002.json
Rate limit headers: not supplied
Selected company-year filings: 28


/tmp/ipykernel_545/1810757003.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sub = pd.concat(parts, ignore_index=True)


,CIK,ticker,company,filing_date,filing_year,accession,primary_document
27,1800,ABT,Abbott Laboratories,2019-02-22,2019,0001047469-19-000624,a2237733z10-k.htm
26,1800,ABT,Abbott Laboratories,2020-02-21,2020,0001104659-20-023904,abt-20191231x10k59d41b.htm
25,1800,ABT,Abbott Laboratories,2021-02-19,2021,0001104659-21-025751,abt-20201231x10k.htm
24,1800,ABT,Abbott Laboratories,2022-02-18,2022,0001104659-22-025141,abt-20211231x10k.htm
23,1800,ABT,Abbott Laboratories,2023-02-17,2023,0001628280-23-004026,abt-20221231.htm
22,1800,ABT,Abbott Laboratories,2024-02-16,2024,0001628280-24-005348,abt-20231231.htm
11,320193,AAPL,Apple Inc.,2019-10-31,2019,0000320193-19-000119,a10-k20199282019.htm
10,320193,AAPL,Apple Inc.,2020-10-30,2020,0000320193-20-000096,aapl-20200926.htm
9,320193,AAPL,Apple Inc.,2021-10-29,2021,0000320193-21-000105,aapl-20210925.htm
8,320193,AAPL,Apple Inc.,2022-10-28,2022,0000320193-22-000108,aapl-20220924.htm


### Download filing documents and estimate Item 1A word counts
This extracts text between Item 1A and Item 1B (or Item 2) from the **primary 10-K document**. SEC filing HTML varies a lot, so this is a starting heuristic, **not a verified research-grade extractor**. It may accidentally find a table-of-contents heading or miss an unusual filing. I flag unusually short sections for manual review and do not silently treat failures as valid zero-length disclosures.

In [5]:
def item_1a_words(document_bytes):
    soup = BeautifulSoup(document_bytes, 'html.parser')
    for element in soup(['script','style','noscript']): element.decompose()
    text = soup.get_text(' ', strip=True)
    text = re.sub(r'\s+', ' ', html.unescape(text))
    start = list(re.finditer(r'(?i)\bitem\s*1\s*a\s*[.\-:–]?\s*risk\s*factors\b', text))
    # Pick the longest plausible section; TOC references are typically very short.
    candidates = []
    for match in start:
        end = re.search(r'(?i)\bitem\s*1\s*b\b|\bitem\s*2\s*[.\-:–]?\s*properties\b', text[match.end():])
        if end:
            section = text[match.end():match.end()+end.start()]
            words = len(section.split())
            if 200 <= words <= 100000: candidates.append((words, section))
    if not candidates: return None, 'needs_manual_review'
    return max(candidates, key=lambda x:x[0])[0], 'heuristic_check_required'

risk_rows = []
for r in filings.itertuples(index=False):
    accession_no_dashes = r.accession.replace('-', '')
    filing_url = f'https://www.sec.gov/Archives/edgar/data/{int(r.CIK)}/{accession_no_dashes}/{r.primary_document}'
    cache = RAW / f'filing_{int(r.CIK)}_{accession_no_dashes}_{Path(r.primary_document).name}'
    try:
        raw_doc = fetch_cached(filing_url, cache)
        count, status = item_1a_words(raw_doc)
    except (RuntimeError, ValueError) as error:
        print('FILING NEEDS REVIEW:', r.ticker, r.filing_year, str(error)[:140])
        count, status = None, 'download_failed'
    risk_rows.append({'CIK':int(r.CIK),'ticker':r.ticker,'filing_year':r.filing_year,
                      'filing_date':r.filing_date,'accession':r.accession,
                      'risk_words':count,'extraction_status':status,'filing_url':filing_url})
risk = pd.DataFrame(risk_rows)
risk.to_csv(PROCESSED/'p3_risk_lengths_pilot.csv',index=False)
display(risk[['ticker','filing_year','risk_words','extraction_status']])

HTTP 200 https://www.sec.gov/Archives/edgar/data/1800/000104746919000624/a2237733z10-k.htm
Rate limit headers: not supplied
HTTP 200 https://www.sec.gov/Archives/edgar/data/1800/000110465920023904/abt-20191231x10k59d41b.htm
Rate limit headers: not supplied
HTTP 200 https://www.sec.gov/Archives/edgar/data/1800/000110465921025751/abt-20201231x10k.htm
Rate limit headers: not supplied
HTTP 200 https://www.sec.gov/Archives/edgar/data/1800/000110465922025141/abt-20211231x10k.htm
Rate limit headers: not supplied
HTTP 200 https://www.sec.gov/Archives/edgar/data/1800/000162828023004026/abt-20221231.htm
Rate limit headers: not supplied
HTTP 200 https://www.sec.gov/Archives/edgar/data/1800/000162828024005348/abt-20231231.htm
Rate limit headers: not supplied
HTTP 200 https://www.sec.gov/Archives/edgar/data/320193/000032019319000119/a10-k20199282019.htm
Rate limit headers: not supplied
HTTP 200 https://www.sec.gov/Archives/edgar/data/320193/000032019320000096/aapl-20200926.htm
Rate limit headers: n

,ticker,filing_year,risk_words,extraction_status
0,ABT,2019,3737,heuristic_check_required
1,ABT,2020,3726,heuristic_check_required
2,ABT,2021,4073,heuristic_check_required
3,ABT,2022,7259,heuristic_check_required
4,ABT,2023,7725,heuristic_check_required
5,ABT,2024,7853,heuristic_check_required
6,AAPL,2019,7924,heuristic_check_required
7,AAPL,2020,8785,heuristic_check_required
8,AAPL,2021,9658,heuristic_check_required
9,AAPL,2022,10220,heuristic_check_required


## 2. Ethics, legality, and what changed from A8
My A8 plan used SEC EDGAR, the cleaned S&P 500 company list from P2, and Stooq prices. I kept the SEC and Wikipedia sources, but I have not confirmed that Stooq permits automated CSV downloading. **This notebook does not scrape Stooq.** To finish the returns side, I need a CSV obtained through an approved source or with permission. I only use public company information, keep the original raw responses, identify my requests, and limit the request rate. I do not bypass restrictions. The pilot has five companies instead of the full ~500-company plan, and I will clearly label that limitation. Wikipedia's current constituents can cause survivorship bias.

## 3. Join design — before the merge
**Grain:** `risk` is one row per (`CIK`, `filing_year`) after choosing one 10-K per filing calendar year. `prices` should be one row per (`ticker`, `date`) with an adjusted closing price. A company-year risk row maps to **two dates** in prices: the first available trading day on or after filing, and the first available trading day on or after 12 calendar months later. This is a time-aligned lookup, not a direct company-year merge.

**Keys and cardinality:** I map ticker symbols to CIK from the same company table. There must be at most one price per ticker/date and one risk row per CIK/year. Price matching is many-to-one for each date lookup, and the final output must remain one row per risk record. **Expected rows:** the number of usable risk rows, with no fan-out. Missing prices remain missing and are flagged, not invented. I use the first trading day **on or after** each target date; the end date is 12 calendar months after the actual filing date, not December 31. Stock returns must use split/dividend-adjusted prices if I want total returns; otherwise they are price returns and must be labeled as such.

### Import approved daily price data
Place a legally obtained file at `data/raw/approved_daily_prices.csv` with columns `ticker,date,adjusted_close`. If your source provides only unadjusted close, rename the column to `close`, and the result will be labeled **price return**, not total return. This notebook does not download prices automatically. **If the file is missing, the risk-text pilot can still run, but the returns join remains incomplete.**

In [6]:
PRICE_FILE = RAW / 'approved_daily_prices.csv'
if PRICE_FILE.exists():
    prices = pd.read_csv(PRICE_FILE)
    required = {'ticker','date'}
    if not required.issubset(prices.columns): raise ValueError('Price CSV needs ticker and date columns')
    price_col = 'adjusted_close' if 'adjusted_close' in prices else 'close'
    if price_col not in prices: raise ValueError('Price CSV needs adjusted_close or close')
    prices['date'] = pd.to_datetime(prices['date'],errors='raise')
    prices['ticker'] = prices['ticker'].astype(str).str.upper().str.strip()
    prices[price_col] = pd.to_numeric(prices[price_col],errors='raise')
    prices = prices[['ticker','date',price_col]].sort_values(['ticker','date'])
    assert not prices.duplicated(['ticker','date']).any(), 'Duplicate ticker/date price records'
    assert prices[price_col].gt(0).all(), 'Prices must be positive'
    print('Loaded',len(prices),'price observations; column:',price_col)
else:
    prices = None
    print('PENDING: add approved_daily_prices.csv to data/raw/ to finish the stock-return join')

PENDING: add approved_daily_prices.csv to data/raw/ to finish the stock-return join


### Join risk changes to the next 12 months of returns
The `merge_asof` lookup finds the next available trading day, but I cap the gap at 7 days so an absent or delisted ticker does not silently match a distant date. I calculate the change in Item 1A words only when the prior filing year is consecutive and both counts are available.

In [7]:
risk = risk.sort_values(['CIK','filing_year']).copy()
risk['prior_year'] = risk.groupby('CIK')['filing_year'].shift()
risk['prior_words'] = risk.groupby('CIK')['risk_words'].shift()
risk['risk_change_pct'] = (risk['risk_words']/risk['prior_words']-1)*100
risk.loc[risk['filing_year'].sub(risk['prior_year']).ne(1) | risk['prior_words'].le(0), 'risk_change_pct'] = float('nan')

if prices is not None:
    base = risk.reset_index(drop=True).copy()
    base['_row_id'] = range(len(base))
    base['target_start'] = pd.to_datetime(base['filing_date'])
    base['target_end'] = base['target_start'] + pd.DateOffset(months=12)
    def next_price(rows, target_col, suffix):
        left = rows[['_row_id','ticker',target_col]].rename(columns={target_col:'target_date'}).sort_values('target_date')
        right = prices.rename(columns={'date':'trade_date',price_col:'price'}).sort_values('trade_date')
        found = pd.merge_asof(left, right, left_on='target_date',right_on='trade_date',by='ticker',direction='forward',tolerance=pd.Timedelta(days=7))
        return found[['_row_id','trade_date','price']].rename(columns={'trade_date':f'{suffix}_trade_date','price':f'{suffix}_price'})
    start = next_price(base,'target_start','start')
    end = next_price(base,'target_end','end')
    joined = base.merge(start,on='_row_id',how='left',validate='one_to_one').merge(end,on='_row_id',how='left',validate='one_to_one')
    joined['return_12m_pct'] = (joined['end_price']/joined['start_price']-1)*100
    joined['return_type'] = 'adjusted-price return' if price_col=='adjusted_close' else 'unadjusted price return'
    joined['price_match_status'] = joined.apply(lambda x: 'matched' if pd.notna(x['return_12m_pct']) else 'missing_price',axis=1)
    joined.to_csv(PROCESSED/'p3_risk_returns_join.csv',index=False)
    display(joined[['ticker','filing_year','risk_words','risk_change_pct','return_12m_pct','price_match_status']])
else:
    joined = None
    print('No returns join yet: approved historical price data is required.')

No returns join yet: approved historical price data is required.


## 4. Verification suite
I use assertions to catch errors rather than assuming a merge is correct. Each comment says what it protects against.

In [8]:
# Prevents one company from appearing twice in the same filing year.
assert not risk.duplicated(['CIK','filing_year']).any(), 'Duplicate company-year filings'

# Protects against invalid negative or zero word counts from extraction.
assert risk['risk_words'].dropna().gt(0).all(), 'Invalid risk-factor word counts'

# Makes sure filing dates fall in the years selected for the pilot.
assert risk['filing_year'].between(START_YEAR,END_YEAR).all(), 'Filing outside the study window'

# Protects against mixing up company identifiers when retrieving filings.
assert set(risk['CIK']).issubset(set(pilot['CIK'])), 'Unknown CIK in risk data'

if joined is not None:
    # Catches fan-out or dropped rows during both price lookups.
    assert len(joined)==len(risk), 'Join changed the number of filing rows'
    # Protects against duplicate company-year records after joining.
    assert not joined.duplicated(['CIK','filing_year']).any(), 'Duplicate company-year after join'
    # Ensures returns are only calculated when both dates have valid prices.
    assert joined.loc[joined['return_12m_pct'].notna(),['start_price','end_price']].notna().all().all(), 'Return without both prices'
    print('PASS: risk checks and returns join checks')
else:
    print('PASS: risk checks. Returns checks pending approved price CSV.')

PASS: risk checks. Returns checks pending approved price CSV.


### Deliberate fan-out test
A duplicate key on the right should raise an error when I say the merge is one-to-one.

In [9]:
left_test = pd.DataFrame({'CIK':[1,2]})
right_test = pd.DataFrame({'CIK':[1,1,2], 'value':[10,11,20]})
try:
    left_test.merge(right_test,on='CIK',validate='one_to_one')
    raise AssertionError('Fan-out was not caught')
except pd.errors.MergeError as e:
    print('PASS: duplicate-key merge correctly rejected:',e)

PASS: duplicate-key merge correctly rejected: Merge keys are not unique in right dataset; not a one-to-one merge


## 5. What surprised me / manual review
One thing I learned is that a company can have more than one ticker but still share the same SEC CIK. Also, pulling the SEC metadata is much easier than reliably identifying Item 1A inside every 10-K. I kept extraction statuses so I can check questionable filings instead of pretending every number is correct.

**Before submitting:** Inspect a few extracted filing sections against the original SEC filing pages, and write down which years were checked and any extraction mistakes. Report the actual count of successful extractions and missing values from the output above. If the approved price file is not present, state clearly that stock-return integration is pending. Do not claim a correlation or result until the data is checked.

## Self-scored rubric (provisional — update after running)

| Criterion | Weight | Score / 2 | Weighted |
|---|---:|---:|---:|
| Acquisition code | ×2 | 1 | 2 |
| Ethics and legality | ×2 | 2 | 4 |
| Join design | ×2 | 1 | 2 |
| Verification | ×3 | 1 | 3 |
| Reproducibility | ×1 | 1 | 1 |
| **Total (provisional)** | | | **12/20** |

I gave myself partial credit because I have the SEC acquisition and risk-text pilot set up, but the extraction still needs manual checks and the stock-return join cannot be verified without approved price data. I used public sources, included rate limiting and caching, explained the join before running it, and wrote assertions to catch problems. I will update these scores based on what actually runs and what I verify.

**Reproducibility:** Install `requests pandas beautifulsoup4 lxml`, enter your contact email, and run all cells. Cache files are saved in `data/raw/`; processed outputs are saved in `data/processed/`. The notebook deliberately does not fetch stock prices until I have a permitted source. The SEC may reject or rate-limit requests; do not bypass access controls.

**AI use note:** I used ChatGPT to help restructure and draft code for this notebook. I reviewed the logic and am responsible for checking the results and complying with course rules.